# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iayamin6/FlyRank-ML-Internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/iayamin6/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.620


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.620   vs   tree 0.720


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Compare depths on the same client holdout used by the reference pipeline.
import importlib.util
from pathlib import Path
from sklearn.metrics import average_precision_score
sys.path.insert(0, str(Path("scripts").resolve()))
spec = importlib.util.spec_from_file_location("reference_training", "scripts/03_train_model.py")
reference_training = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference_training)
train_idx, test_idx, split_kind = reference_training.make_client_aware_split(
    df, df["is_declining_label"])
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
assert split_kind == "client_holdout" and train_clients.isdisjoint(test_clients)
assert not {"trend_pct", "trend_direction", "client_id", "content_id"}.intersection(features)
print(f"{len(train_idx):,} training pages / {len(test_idx):,} held-out pages")
print(f"{len(train_clients)} training clients / {len(test_clients)} held-out clients; overlap = 0")
print(f"Held-out declining prevalence: {y[test_idx].mean():.3f}")

# Keep starter preprocessing to isolate depth and split effects.
# Stable input order makes tie-breaking reproducible; also report its expectation.
def ranking_metrics(scores, labels, k=50):
    scores, labels = np.asarray(scores), np.asarray(labels)
    order = np.argsort(-scores, kind="stable")
    cutoff = scores[order[k - 1]]
    above, tied = scores > cutoff, scores == cutoff
    slots = k - above.sum()
    expected = (labels[above].sum() + slots * labels[tied].mean()) / k
    return {"p50_stable": labels[order[:k]].mean(),
            "p50_random_tie_expectation": expected,
            "average_precision": average_precision_score(labels, scores),
            "cutoff_tie_pages": int(tied.sum())}

comparison_rows = []
heldout_scores = {"hand rule": df.iloc[test_idx]["hand_rule_score"].to_numpy()}
for depth in (2, 3, 4):
    candidate = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    candidate.fit(X.iloc[train_idx], y[train_idx])
    heldout_scores[f"tree depth {depth}"] = candidate.predict_proba(X.iloc[test_idx])[:, 1]
    print(f"Depth {depth}: {candidate.get_n_leaves()} leaves; root = {features[candidate.tree_.feature[0]]}")
    if depth == 3:
        readable_tree = candidate
for name, scores in heldout_scores.items():
    comparison_rows.append({"ranking": name, **ranking_metrics(scores, y[test_idx])})
comparison = pd.DataFrame(comparison_rows).set_index("ranking")
display(comparison.round(4))
print("Depth-3 learned rules (trained only on training clients):")
print(export_text(readable_tree, feature_names=features))

# Commit only aggregate evaluation receipts, never held-out rows or IDs.
import json
receipt_dir = Path("work/outputs")
receipt_dir.mkdir(parents=True, exist_ok=True)
receipt = {"seed": 42, "split": split_kind,
           "train_pages": len(train_idx), "test_pages": len(test_idx),
           "train_clients": len(train_clients), "test_clients": len(test_clients),
           "test_prevalence": float(y[test_idx].mean()),
           "features": features,
           "metrics": comparison.reset_index().to_dict(orient="records")}
(receipt_dir / "assignment01_holdout_metrics.json").write_text(json.dumps(receipt, indent=2))
print("Saved aggregate evaluation receipt.")

27,675 training pages / 2,325 held-out pages
26 training clients / 6 held-out clients; overlap = 0
Held-out declining prevalence: 0.391
Depth 2: 4 leaves; root = impressions_90d
Depth 3: 8 leaves; root = impressions_90d
Depth 4: 16 leaves; root = impressions_90d


,p50_stable,p50_random_tie_expectation,average_precision,cutoff_tie_pages
ranking,,,,
hand rule,0.40,0.3910,0.3910,2325
tree depth 2,0.52,0.5142,0.5060,1439
tree depth 3,0.62,0.5600,0.5329,916
tree depth 4,0.58,0.5367,0.5289,613


Depth-3 learned rules (trained only on training clients):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.90
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.90
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 0
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

Saved aggregate evaluation receipt.


### Evaluation choices and limitations
Depths 2, 3, and 4 are predefined exploratory comparisons, not a model-selection search with an untouched final test. The stable Precision@50 uses source-row order to break ties; the expected value averages uniformly over the tied boundary group without selecting favorable labels. Average precision assesses the broader ranking and handles tied score groups together.

The reference zero-fill is retained to isolate the depth/split comparison. It has limitations: missing word counts can encode content type, and zero position means unavailable position. A later model should learn imputation only from training data and include explicit missingness indicators.

Client holdout prevents client overlap, but these snapshot features overlap the period used to define decline. These results describe observed decline on held-out clients; they do not validate future decline prediction or causal refresh benefits. The deliberately leaky teaching tree above is excluded from the comparison.

**Observed in this run:** the 6 held-out clients contain 2,325 pages (39.10% declining), versus 27,675 pages from 26 training clients. The hand rule assigns every held-out page zero, so its stable Precision@50 of 0.400 reflects arbitrary row order; its random-tie expectation is the 0.3910 prevalence. This is a coverage failure of the rule on this holdout, not useful ranking discrimination.

Depths 2/3/4 yield stable Precision@50 of 0.520/0.620/0.580 and random-tie expectations of 0.5142/0.5600/0.5367. Depth 3 improves on depth 2 here, but depth 4 does not; the depth-3 tree remains readable with eight leaves and splits first on impressions. Its average precision is 0.5329, compared with 0.3910 for the constant hand-rule ranking. The 916-page tie at the depth-3 cutoff explains why reporting only its 0.620 point estimate would overstate the precision of this comparison.

The original in-sample lesson has a different comparison: the hand rule wins at 20 (0.900 versus 0.800), while the depth-2 tree wins at 50 (0.720 versus 0.620). The leaky tree's 1.000 is an outcome-reconstruction demonstration, not valid predictive performance.

In [8]:
# Record the execution environment alongside the actual outputs.
import platform
from importlib.metadata import version
print("Python", platform.python_version())
for package in ("numpy", "pandas", "scikit-learn", "matplotlib", "reportlab"):
    print(package, version(package))

Python 3.13.0
numpy 2.5.4
pandas 3.0.6
scikit-learn 1.9.1
matplotlib 3.11.2
reportlab 5.0.1


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.